# Week 3: Edge and Contour Detection
## Complete Coin Counting Experiment

This notebook demonstrates the complete process from simple high-frequency extraction to the full Canny edge detection algorithm, culminating in automatic coin counting.

**Learning Objectives:**
- Understand the fundamental principles of edge detection
- Master edge detection operators such as Sobel, Laplacian, and LoG
- Understand the complete Canny edge detection pipeline
- Learn to use contour detection for object counting

**Required Libraries:**
- OpenCV (cv2)
- NumPy
- Matplotlib (for image display)

## 1. Environment Setup and Library Import

In [ ]:
%pip install -q opencv-python

import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib import rcParams

# Configure Matplotlib to support Chinese display
rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']  # macOS/Windows
rcParams['axes.unicode_minus'] = False  # Fix minus sign display

# Set random seed (ensures reproducible results)
np.random.seed(42)

# Output directory
OUTPUT_DIR = "generated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Environment setup complete!")

### Helper Functions: Image Display and Saving

In [ ]:
def save_img(filename, img):
    """Save image to output directory"""
    path = os.path.join(OUTPUT_DIR, filename)
    cv2.imwrite(path, img)
    print(f"  Saved: {filename}")

def show_images(images, titles, figsize=(15, 5), cmap='gray'):
    """Display multiple images side by side in notebook"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3 and img.shape[2] == 3:
            # BGR to RGB for color images
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap=cmap)
        axes[i].set_title(title, fontsize=12)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

print("Helper functions defined!")

## 2. Load and Preprocess Image

In [ ]:
# Load coin image
asset_path = "asset_coins_simple.png"
coins_gray_full = cv2.imread(asset_path, cv2.IMREAD_GRAYSCALE)
coins_color_full = cv2.imread(asset_path)

# Resize to half size for easier processing and display
h_full, w_full = coins_gray_full.shape
scale = 0.5
new_w, new_h = int(w_full * scale), int(h_full * scale)

coins_clean = cv2.resize(coins_gray_full, (new_w, new_h), interpolation=cv2.INTER_AREA)
coins_color = cv2.resize(coins_color_full, (new_w, new_h), interpolation=cv2.INTER_AREA)

h, w = coins_clean.shape
print(f"Original resolution: {w_full}x{h_full}")
print(f"Working resolution: {w}x{h}")

# Display original image
show_images([coins_clean, coins_color], 
            ['Grayscale', 'Color'])

---
## Experiment 0: Generate Noisy Image

To demonstrate the importance of Gaussian smoothing in the Canny algorithm, we need a noisy image.

In [ ]:
# Add Gaussian noise
sigma_noise = 20
noise = np.random.normal(0, sigma_noise, coins_clean.shape)
coins_noisy = np.clip(coins_clean.astype(np.float32) + noise, 0, 255).astype(np.uint8)

save_img("asset_coins_noisy.png", coins_noisy)

# Display comparison
show_images([coins_clean, coins_noisy], 
            [f'Original (Clean)', f'With Noise (σ={sigma_noise})'])

print(f"Noise parameter: σ = {sigma_noise}")

---
## Experiment 1: Gaussian Blur + High-Frequency Extraction
### Review of Week 2 Knowledge

**Principle:**
- High-frequency component = Original image - Low-frequency component
- Low-frequency component can be obtained through Gaussian blur
- High-frequency component contains edge information, but not precise enough

In [ ]:
# Gaussian blur to extract low frequency
blurred_exp1 = cv2.GaussianBlur(coins_clean, (25, 25), 0)

# Extract high frequency: |Original - Blurred|
high_freq_exp1 = cv2.absdiff(coins_clean, blurred_exp1)
high_freq_exp1 = cv2.normalize(high_freq_exp1, None, 0, 255, cv2.NORM_MINMAX)

# Save images
save_img("page_003_img_original_cv.png", coins_clean)
save_img("page_003_img_blurred_cv.png", blurred_exp1)
save_img("page_003_img_high_freq_cv.png", high_freq_exp1)

# Display results
show_images([coins_clean, blurred_exp1, high_freq_exp1],
            ['Original', 'Gaussian Blur (Low Frequency)', 'High Frequency (Edges)'])

**Observation:** The high-frequency image contains edge information, but the edges are thick and contain many texture details.

---
## Experiment 2: High Frequency + Contour Detection (Failed Case)
### Why isn't the Week 2 method good enough?

Try using the high-frequency image directly for contour detection and see what happens.

In [ ]:
# Extract high frequency
blurred_exp2 = cv2.GaussianBlur(coins_clean, (15, 15), 0)
high_freq_raw = cv2.absdiff(coins_clean, blurred_exp2)
high_freq_exp2 = cv2.normalize(high_freq_raw, None, 0, 255, cv2.NORM_MINMAX)

# Binarize
_, binary_exp2 = cv2.threshold(high_freq_raw, 20, 255, cv2.THRESH_BINARY)

# Find contours (no filtering)
contours_exp2, _ = cv2.findContours(binary_exp2, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours on color image
result_exp2 = coins_color.copy()
cv2.drawContours(result_exp2, contours_exp2, -1, (0, 0, 255), 2)
count_text = f"Count: {len(contours_exp2)}"
cv2.putText(result_exp2, count_text, (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 5)
cv2.putText(result_exp2, count_text, (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)

save_img("page_008_img_high_freq_cv.png", high_freq_exp2)
save_img("page_008_img_contours_cv.png", result_exp2)

show_images([high_freq_exp2, binary_exp2, result_exp2],
            ['High Frequency', 'Binary', f'Contour Detection: {len(contours_exp2)} contours'],
            figsize=(18, 5))

print(f"Detected contours: {len(contours_exp2)}")
print(f"Actual number of coins: 12")
print("❌ Failed! Too many contours detected")

**Problem Analysis:**
- High-frequency image contains too many details (coin surface texture, lighting changes, etc.)
- Each detail can form an independent contour
- Total contours far exceed the actual number of coins

**Conclusion:** Need a more precise edge detection method!

---
## Experiment 3: Sobel Operator Directionality
### First-Order Derivatives: Detecting Edge Direction

**Sobel Operator:**
- Sobel X: Detect vertical edges (horizontal gradient)
- Sobel Y: Detect horizontal edges (vertical gradient)
- Gradient magnitude: Combine both directions to get complete edges

In [ ]:
# First use Gaussian blur to suppress coin surface texture
pre_blurred = cv2.GaussianBlur(coins_clean, (31, 31), 0)

# Sobel X: Detect vertical edges
gx = cv2.Sobel(pre_blurred, cv2.CV_64F, 1, 0, ksize=3)
gx_vis = cv2.normalize(np.abs(gx), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Sobel Y: Detect horizontal edges
gy = cv2.Sobel(pre_blurred, cv2.CV_64F, 0, 1, ksize=3)
gy_vis = cv2.normalize(np.abs(gy), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Gradient magnitude: √(Gx² + Gy²)
magnitude = np.sqrt(gx**2 + gy**2)
magnitude = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Save images
save_img("page_022_img_original_cv.png", coins_clean)
save_img("page_022_img_gx_cv.png", gx_vis)
save_img("page_022_img_gy_cv.png", gy_vis)
save_img("page_022_img_magnitude_cv.png", magnitude)

# Display results
show_images([coins_clean, gx_vis, gy_vis, magnitude],
            ['Original', 'Sobel X\n(Vertical Edges)', 'Sobel Y\n(Horizontal Edges)', 'Magnitude\n(Complete Edges)'],
            figsize=(20, 5))

**Observation:**
- Sobel X detects the left and right edges of coins (vertical edges)
- Sobel Y detects the top and bottom edges of coins (horizontal edges)
- Gradient magnitude combines both directions to get complete circular edges

---
## Experiment 4: Sobel vs High-Frequency
### Advantages of the Week 3 Method

Compare the high-frequency extraction from Week 2 and the Sobel operator from Week 3.

In [ ]:
# Week 2 method: High frequency = |Original - Gaussian Blur|
blurred_hf = cv2.GaussianBlur(coins_clean, (15, 15), 0)
high_freq_exp4 = cv2.absdiff(coins_clean, blurred_hf)
high_freq_exp4 = cv2.normalize(high_freq_exp4, None, 0, 255, cv2.NORM_MINMAX)

# Week 3 method: Sobel gradient magnitude
gx_exp4 = cv2.Sobel(coins_clean, cv2.CV_64F, 1, 0, ksize=3)
gy_exp4 = cv2.Sobel(coins_clean, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(gx_exp4**2 + gy_exp4**2)
sobel_mag = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Save images
save_img("page_023_img_high_freq_cv.png", high_freq_exp4)
save_img("page_023_img_sobel_cv.png", sobel_mag)

# Display comparison
show_images([high_freq_exp4, sobel_mag],
            ['Week 2: High Frequency', 'Week 3: Sobel Gradient'],
            figsize=(12, 5))

**Comparison Analysis:**
- **High-Frequency Image**: Thick edges, contains many texture details
- **Sobel Image**: Thinner and more precise edges, better for subsequent processing

---
## Experiment 5: Laplacian vs LoG
### Second-Order Derivatives: Why Smooth First?

**Laplacian (Laplace Operator):**
- Second-order derivative, extremely sensitive to noise
- LoG (Laplacian of Gaussian): Gaussian smoothing first, then Laplacian

Use a noisy image to demonstrate the difference.

In [ ]:
# Direct Laplacian (sensitive to noise)
lap = cv2.Laplacian(coins_noisy, cv2.CV_64F, ksize=3)
lap_vis = cv2.normalize(np.abs(lap), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# LoG: Gaussian smoothing first, then Laplacian
blurred_log = cv2.GaussianBlur(coins_noisy, (7, 7), 1.0)
log = cv2.Laplacian(blurred_log, cv2.CV_64F, ksize=3)
log_vis = cv2.normalize(np.abs(log), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Save images
save_img("page_027_img_laplacian_cv.png", lap_vis)
save_img("page_027_img_log_cv.png", log_vis)

# Display comparison
show_images([coins_noisy, lap_vis, log_vis],
            ['Noisy Original', 'Direct Laplacian\n(Noise Amplified)', 'LoG\n(Smooth then Derivative)'],
            figsize=(15, 5))

**Important Conclusion:**
- Second-order derivatives are extremely sensitive to noise
- **Must smooth before taking derivatives**
- This is the theoretical basis for the first step of the Canny algorithm

---
## Experiments 6-7: First Two Steps of Canny Algorithm
### Step 1: Gaussian Smoothing for Noise Reduction
### Step 2: Gradient Calculation

In [ ]:
# Step 1: Gaussian smoothing
smoothed = cv2.GaussianBlur(coins_noisy, (5, 5), 1.4)

# Step 2: Calculate gradient
gx_canny = cv2.Sobel(smoothed, cv2.CV_64F, 1, 0, ksize=3)
gy_canny = cv2.Sobel(smoothed, cv2.CV_64F, 0, 1, ksize=3)
gradient = np.sqrt(gx_canny**2 + gy_canny**2)
gradient_vis = cv2.normalize(gradient, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Save images
save_img("page_032_img_noisy_cv.png", coins_noisy)
save_img("page_032_img_smoothed_cv.png", smoothed)
save_img("page_033_img_smoothed_cv.png", smoothed)
save_img("page_033_img_gradient_cv.png", gradient_vis)

# Display first two steps of Canny
show_images([coins_noisy, smoothed, gradient_vis],
            ['Original Noisy Image', 'Step 1: Gaussian Smoothing', 'Step 2: Gradient Calculation'],
            figsize=(15, 5))

**Canny Algorithm Pipeline:**
1. ✅ **Gaussian Smoothing**: Remove noise
2. ✅ **Gradient Calculation**: Using Sobel operator
3. ⏳ **Non-Maximum Suppression (NMS)**: Edge thinning
4. ⏳ **Double Threshold + Edge Tracking**: Determine final edges

---
## Experiment 8: Non-Maximum Suppression (NMS)
### Step 3: Edge Thinning

**Purpose of NMS:** Thin thick edges to single-pixel-wide precise edges.

In [ ]:
# Before NMS: Sobel gradient (thick edges)
blurred_nms = cv2.GaussianBlur(coins_noisy, (5, 5), 1.4)
gx_nms = cv2.Sobel(blurred_nms, cv2.CV_64F, 1, 0, ksize=3)
gy_nms = cv2.Sobel(blurred_nms, cv2.CV_64F, 0, 1, ksize=3)
before_nms = np.sqrt(gx_nms**2 + gy_nms**2)
before_nms = cv2.normalize(before_nms, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# After NMS: Canny result (thin edges)
after_nms = cv2.Canny(blurred_nms, 50, 150)

# Save images
save_img("page_038_img_before_cv.png", before_nms)
save_img("page_038_img_after_cv.png", after_nms)

# Display comparison
show_images([before_nms, after_nms],
            ['Before NMS: Thick Edges', 'After NMS: Thin Edges'],
            figsize=(12, 5))

**Observation:**
- Before NMS: Edges have certain width, not precise enough
- After NMS: Edges become single-pixel wide, very precise

---
## Experiment 9: Importance of Double Threshold
### Step 4: Determine Final Edges

**Double Threshold Strategy:**
- **High Threshold**: Strong edges (definitely edges)
- **Low Threshold**: Weak edges (possibly edges)
- **Edge Tracking**: Retain weak edges connected to strong edges

In [ ]:
blurred_thresh = cv2.GaussianBlur(coins_noisy, (5, 5), 1.4)

# Threshold too high: Edge discontinuity
canny_high = cv2.Canny(blurred_thresh, 100, 200)

# Double threshold appropriate: Complete edges
canny_dual = cv2.Canny(blurred_thresh, 50, 150)

# Threshold too low: Too much noise
canny_low = cv2.Canny(blurred_thresh, 10, 50)

# Save images
save_img("page_040_img_high_cv.png", canny_high)
save_img("page_040_img_dual_cv.png", canny_dual)
save_img("page_040_img_low_cv.png", canny_low)

# Calculate edge pixel counts
px_high = np.sum(canny_high > 0)
px_dual = np.sum(canny_dual > 0)
px_low = np.sum(canny_low > 0)

# Display comparison
show_images([canny_high, canny_dual, canny_low],
            [f'Threshold Too High (100, 200)\nEdge Pixels: {px_high}', 
             f'Threshold Moderate (50, 150)\nEdge Pixels: {px_dual}',
             f'Threshold Too Low (10, 50)\nEdge Pixels: {px_low}'],
            figsize=(15, 5))

print(f"High threshold edge pixels: {px_high}")
print(f"Moderate threshold edge pixels: {px_dual}")
print(f"Low threshold edge pixels: {px_low}")

**Threshold Selection Principles:**
- Too high threshold → Discontinuous edges, loss of important information
- Too low threshold → Pseudo-edges from noise
- **Rule of Thumb**: High Threshold = 2-3 × Low Threshold

---
## Experiment 10: Necessity of Gaussian Smoothing
### Proof of Importance of Canny's First Step

Compare Canny results with and without Gaussian smoothing.

In [ ]:
# Without Gaussian smoothing: Direct Canny
canny_without = cv2.Canny(coins_noisy, 50, 150)

# With Gaussian smoothing: Smooth then Canny
blurred_final = cv2.GaussianBlur(coins_noisy, (5, 5), 1.4)
canny_with = cv2.Canny(blurred_final, 50, 150)

# Save images
save_img("page_048_img_without_cv.png", canny_without)
save_img("page_048_img_with_cv.png", canny_with)

# Calculate edge pixels
px_without = np.sum(canny_without > 0)
px_with = np.sum(canny_with > 0)
ratio = px_without / px_with

# Display comparison
show_images([coins_noisy, canny_without, canny_with],
            ['Noisy Original Image', 
             f'Without Gaussian Smoothing\nEdge Pixels: {px_without}',
             f'With Gaussian Smoothing\nEdge Pixels: {px_with}'],
            figsize=(15, 5))

print(f"Edge pixels without Gaussian: {px_without}")
print(f"Edge pixels with Gaussian: {px_with}")
print(f"Noise increases edge pixels: {ratio:.1f}x")

**Conclusion:**
- Without Gaussian smoothing, noise produces many pseudo-edges
- Gaussian smoothing is a necessary step in the Canny algorithm

---
## Experiment 11: Edges vs Contours
### From Pixels to Curves

**Concept Distinction:**
- **Edge (Edge)**: Binary image, each pixel marked whether it is an edge
- **Contour (Contour)**: Continuous point sequence, describes a closed curve

In [ ]:
# Edge image: Pixel-level marking
edges_exp11 = cv2.Canny(coins_clean, 50, 150)

# Contour image: Continuous curves
contours_exp11, _ = cv2.findContours(edges_exp11.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw contours on color image
colors = [(0, 255, 0), (255, 0, 0), (0, 0, 255),
          (255, 255, 0), (255, 0, 255), (0, 255, 255)]
result_contour = coins_color.copy()
for i, cnt in enumerate(contours_exp11):
    if cv2.contourArea(cnt) > 500:  # Only draw large contours
        cv2.drawContours(result_contour, [cnt], -1, colors[i % len(colors)], 2)

# Save images
save_img("page_052_img_edge_cv.png", edges_exp11)
save_img("page_053_img_contour_cv.png", result_contour)
save_img("page_054_img_edge_cv.png", edges_exp11)
save_img("page_054_img_contour_cv.png", result_contour)

# Display comparison
show_images([edges_exp11, result_contour],
            ['Edge Image (Pixel Marking)', 'Contour Image (Continuous Curves)'],
            figsize=(12, 5))

print(f"Found {len(contours_exp11)} contours")
print(f"Contours with area > 500: {sum(1 for c in contours_exp11 if cv2.contourArea(c) > 500)} contours")

**Edges vs Contours:**
- Edge detection → Get binary image
- Contour finding → Organize edge pixels into continuous curves
- Contours can be used for: Measuring area, perimeter, shape analysis, etc.

---
## Experiment 12: Comparison of Three Methods
### Evolution from Failure to Success

Show the effectiveness differences of three methods:
1. **Method 1**: High Frequency + Contour (Failed)
2. **Method 2**: Canny + No Filtering (Improved but Inaccurate)
3. **Method 3**: Canny + Filtering (Success)

In [ ]:
# ----- Method 1: High Frequency + Contour (FAIL) -----
blurred_m1 = cv2.GaussianBlur(coins_clean, (15, 15), 0)
high_freq_raw_m1 = cv2.absdiff(coins_clean, blurred_m1)
_, binary_m1 = cv2.threshold(high_freq_raw_m1, 20, 255, cv2.THRESH_BINARY)
contours_m1, _ = cv2.findContours(binary_m1, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

result_m1 = coins_color.copy()
cv2.drawContours(result_m1, contours_m1, -1, (0, 0, 255), 2)
cv2.putText(result_m1, f"Count: {len(contours_m1)}", (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 5)
cv2.putText(result_m1, f"Count: {len(contours_m1)}", (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
cv2.putText(result_m1, "X", (w - 80, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 0), 6)
cv2.putText(result_m1, "X", (w - 80, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 255), 4)

save_img("page_077_img_method1_cv.png", result_m1)

print(f"Method 1: {len(contours_m1)} contours (Failed)")

In [ ]:
# ----- Method 2: Canny + No Filtering (PARTIAL) -----
blurred_m2 = cv2.GaussianBlur(coins_clean, (9, 9), 2.0)
edges_m2 = cv2.Canny(blurred_m2, 50, 150)
contours_m2, _ = cv2.findContours(edges_m2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

result_m2 = coins_color.copy()
cv2.drawContours(result_m2, contours_m2, -1, (0, 0, 255), 2)
cv2.putText(result_m2, f"Count: {len(contours_m2)}", (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
cv2.putText(result_m2, "X", (w - 80, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 255), 4)

save_img("page_077_img_method2_cv.png", result_m2)

print(f"Method 2: {len(contours_m2)} contours (Improved but Inaccurate)")

In [ ]:
# ----- Method 3: Canny + Filtering (SUCCESS) -----
blurred_m3 = cv2.GaussianBlur(coins_clean, (9, 9), 2.0)
edges_m3 = cv2.Canny(blurred_m3, 50, 150)
contours_m3, _ = cv2.findContours(edges_m3, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Filtering: Area range + Circularity threshold
total_pixels = h * w
min_area = int(total_pixels * 0.002)  # 0.2%
max_area = int(total_pixels * 0.15)   # 15%
circularity_threshold = 0.7

valid_m3 = []
for cnt in contours_m3:
    area = cv2.contourArea(cnt)
    if min_area < area < max_area:
        perimeter = cv2.arcLength(cnt, True)
        if perimeter > 0:
            circularity = 4 * np.pi * area / (perimeter ** 2)
            if circularity > circularity_threshold:
                valid_m3.append(cnt)

result_m3 = coins_color.copy()
cv2.drawContours(result_m3, valid_m3, -1, (0, 255, 0), 3)
cv2.putText(result_m3, f"Count: {len(valid_m3)}", (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

save_img("page_077_img_method3_cv.png", result_m3)

print(f"Method 3: {len(valid_m3)} coins")
print(f"Filtering Criteria:")
print(f"  - Area Range: {min_area} ~ {max_area} pixels")
print(f"  - Circularity Threshold: {circularity_threshold}")

In [ ]:
# Display three methods side by side
show_images([result_m1, result_m2, result_m3],
            [f'Method 1: High Freq+Contour\n{len(contours_m1)} coins',
             f'Method 2: Canny+No Filtering\n{len(contours_m2)} coins',
             f'Method 3: Canny+Filtering\n{len(valid_m3)} coins'],
            figsize=(18, 5))

**Progressive Improvement Process:**
1. **Method 1**: High-frequency image too coarse, produces many false contours
2. **Method 2**: Canny edges are precise, but still has small fragments
3. **Method 3**: Add shape filtering (area + circularity), accurately identify coins

---
## Experiment 13: Preliminary Detection Result
### Detected 12 coins, but missed 1

Upon careful observation, there are actually **13 coins**, but our detection only identified **12**.

**Problem Analysis**: The Roman coin in the middle of the top row has complex surface texture (portrait, text), which causes:
- Canny detects many internal edges
- Outer contour is relatively weak or fragmented
- `findContours` cannot form a complete closed contour

In [ ]:
# Use Method 3 results and add number annotations
result_final = coins_color.copy()

for i, cnt in enumerate(valid_m3):
    # Draw contour
    cv2.drawContours(result_final, [cnt], -1, (0, 255, 0), 3)
    
    # Calculate centroid
    M = cv2.moments(cnt)
    if M["m00"] > 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        
        # Add number annotation
        text = str(i + 1)
        cv2.putText(result_final, text, (cx - 10, cy + 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4)
        cv2.putText(result_final, text, (cx - 10, cy + 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

# Display count
cv2.putText(result_final, f"Count: {len(valid_m3)}", (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 5)
cv2.putText(result_final, f"Count: {len(valid_m3)}", (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

save_img("page_079_img_success_cv.png", result_final)

# Display final result
show_images([result_final],
            [f'Preliminary Result: Detected {len(valid_m3)} Coins'],
            figsize=(10, 8))

print(f"\n=== Preliminary Detection Complete ===")
print(f"Detected {len(valid_m3)} coins (Actual: 13, Missed 1)")